In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy.interpolate import interp1d
mpl.rcParams['figure.dpi'] = 300

#### Base Functions

In [ ]:
def rest_freq(n):
    '''
    Calculates H transition frequency in MHz
    ------------------
    n            (int)   : Transition level
    ------------------
    return [MHz] (float) : Frequency of n transition
    '''
    return(((1 / (n)**2) - (1 / (n+1)**2)) * 1.1e5 * 3e10 * 1e-6)

In [ ]:
def freq_z(nu_emit, z):
    '''
    Calculates cosmological redshift of ground state H frequency
    ------------------
    nu_emit (float) : Ground state frequency of transition
    z       (array) : Redshift space of 21cm emission
    ------------------
    return  (array) : Redshifted frequency of H transition
    '''
    return(nu_emit/(1+z))

In [ ]:
def freqtoz(x):
    return((1420 / x) - 1)
def ztofreq(x):
    return (1420 / (1+x))

In [ ]:
def meanT_shifted(tau_L, N_HII, nu_emit, linewidth, bandwidth, z):
    '''
    Calculates the Line temperature of each RRL at the associated redshift. This function is only passed those RRLs (and associated redshifts) that are observable by 
    the 21cm observing experiment.
    ---------------------------------------
    E_SFR     [Solar Mass / yr / Mpc^3] (float) : Comoving Star Formation Rate
    tau_L                               (float) : Optical Depth of RRL in HII region
    N_HII                               (float) : Number of HII obscuring regions w/ optical depth, tau_L
    nu_emit   [MHz]                     (float) : Rest frame frequency of RRL
    linewidth [km / s]                  (float) : Linewidth of RRL, produced by HII region
    bandwidth [MHz]                     (float) : Bandwidth of 21cm observation
    z                                   (array) : Redshift where RRL is observable
    --------------------------------------
    return    [K]                       (array) : Line temperature of particular RRL
    '''
    E_SFR = madauFit(z)
    nu_shift = freq_z(nu_emit, z)
    #T_L = 0.0000509244 * (E_SFR / 0.1) * ((tau_L * N_HII) / 0.1) * (nu_shift / 150)**(-2.8) * (linewidth / 20) * ((1+z)/3)**(-2.8)
    #T_L = 0.00173539 * (E_SFR / 0.1) * ((tau_L * N_HII) / 0.1) * (nu_shift / 150)**(-2.8) * (linewidth / 100) * ((1+z)/3)**(-2.3)
    #T_L = 0.00173539 * (E_SFR / 0.1) * ((tau_L * N_HII) / 0.1) * (nu_shift / 150)**(-2.8) * (linewidth / 20) * (1+z)**(-0.8)
    T_L = 3.5e-4 * (1/np.sqrt(0.27 * (1+z)**3)) * (E_SFR / 0.1) * ((tau_L * N_HII) / 0.1) * (nu_shift / 150)**(-2.8) * (linewidth / 20) * (1+z)**(-0.8)
    return(T_L)

In [ ]:
def madauFit(z):
    '''
    Calculates the fit for SFR from redshift (Madau & Dickinson 2014)
    -----------------------------------
    z                                 (array) : Redshift 
    -----------------------------------
    returns [Solar Mass / yr / Mpc^3] (array) : Star formation rate Density
    '''
    return (0.015 * (1 + z)**2.7 / (1 + ((1 + z)/2.9)**5.6))

In [ ]:
#Returns the 21cm brightness temperature in Kelvin from a homogeneous universe with xHI=0.01
def T21(z, xHI = .01): #xHI chooses expectation of post reioniation if not specified
    return 0.009*(1+z)**(1/2)*xHI  #eqn 18 in https://arxiv.org/pdf/astro-ph/0608032

In [ ]:
def tau_C(Te, n, EM):
    #Formula from blue book eq 2.95
    return(0.08235 * Te**(-1.35) * (rest_freq(n)/1000)**(-2.1) * EM)

#### Model

In [ ]:
def model_plotting(HI_band_min, HI_band_max, N = 10_000, n_min = 1, n_max = 450, tau = 0.1):
    '''
    Base 'model' calculation function:
    Underlying function takes frequency band and calculates total temperature for each transition. Steps of this calculation are as follows:
    1. Choose n transition and calculates rest frame
    2. Shifts rest frame frequency across z = 0 - 6
    3. Identify all frequencies (redshifts) where rest frame frequency falls in observing band
    4. Calculates temperature for that n at all redshifts, normalizing by total number of elements in (3)
    -----------------------------------
    HI_band_min [MHz] (float) : Lower end of frequency band of interest
    HI_band_max [MHz] (float) : Upper end of frequency band of interest
    -----------------------------------
    returns [K] (float) : Total RRL temperature of contamination band
    '''
    #good
    n_range = np.arange(n_min, n_max, 1)
    #z_space = freqtoz(np.linspace(1420, 200, N))
    z_space = np.linspace(0, 6, N)
    z_spacelarge = np.linspace(-6, 12, 3*N)
    #bwdth = HI_band_max - HI_band_min
    bwdth = 1220
    bandwidth_size = ((z_space < freqtoz(HI_band_min)) & (z_space >= freqtoz(HI_band_max))).sum()
    contam_arr = []
    #tau = complexTau(ztofreq(z_space))
    #print(tau)
    for n in n_range:
        #Finds rest frame frequency of each n value
        nu_emitted = rest_freq(n)
        #Shifts the rest frame frequency of the transition by all possible redshifts
        nu_shifted = freq_z(nu_emitted, z_space)
        nu_shifted2 = freq_z(nu_emitted, z_spacelarge)
        #Creates a mask for the shifted frequency to fall within the observation band
        nu_obs_mask = (nu_shifted <= HI_band_max) & (nu_shifted >= HI_band_min)  #& (nu_shifted <= 1420) & (nu_shifted >= 200) #commented this out as wasn't sure what it was doing
        if np.sum(nu_obs_mask ) == 0:
            continue
        bandwidth_size = np.sum((nu_shifted2 <= HI_band_max) & (nu_shifted2 >= HI_band_min))  #& (nu_shifted <= 1420) & (nu_shifted >= 200) #commented this out as wasn't sure what it was doing
        
        #if np.sum(nu_obs_mask)/bandwidth_size  != 0:
        #    print("fraction of bandwidth filled = ", np.sum(nu_obs_mask)/bandwidth_size)
        #Calculates the temperature of that line at all the different redshifts, normalized by bandwidth
        temp_line = meanT_shifted(tau, 1, nu_emitted, 20, 1, z_space) / (bandwidth_size)#(nu_obs_mask.sum() +1e-5)  #I took out the 1e-5
        #Masks out all the temperatures that fall in the observing band
        #T_21 = (T21(z_space))[nu_obs_mask]
        T_21 = lt.T21(z_space)[nu_obs_mask]
        temp_line = temp_line[nu_obs_mask]
        #Combines all the temperatures for that specific line
        contam_arr.append(np.sum(temp_line/T_21))
    #Combines all line temperatures
    return(np.sum(contam_arr))

#### Analytical

In [ ]:
def aFcn2(nu, n_min = 1, n_max = 450, tau_new=0.1):
    '''
    Base 'analytical' calculation function:
    Underlying function takes single frequency and calculates total temperature contamination (from all n) at that frequency. Steps of this calculation are as follows:
    
    1. Find rest frame of each n
    2. Find redshift where n reaches frequency of interest
    3. Calculates temperature of n at calculated redshift
    4. For all n, if corresponding redshift is not 0 < z < 6, sets temperature to 0.
    5. Adds temperature for all remaining n
    -----------------------------------
    nu [MHz] (float) : Discrete frequency of interest
    -----------------------------------
    returns [K] (float) : Total RRL temperature at frequency, nu
    '''
    
    n_range = np.arange(n_min, n_max, 1)
    
    #Find rest frame frequency for each n transition
    nu_emitted = rest_freq(n_range)

    #Calculate redshift where each n would be shifted to given nu
    z_contam = (nu_emitted / nu) - 1
    
    #Creates a mask for all redshifts outside of observable band
    z_mask = (z_contam >= 0) & (z_contam <= 6)

    #Calculates temperature for each transition
    temp_line = lt.meanT_shifted(tau_new, 1, nu_emitted, 20, 1, z_contam)
    
    #Masks out transitions that fall outside of z = (0,6) band
    temp_line = np.where(z_mask, temp_line, 0)

    T_21 = lt.T21(z_contam)

    #Combines all n temperatures for given nu
    return(np.sum(temp_line/T_21))

In [ ]:
def aFcn(nu, N = 10_000, n_min = 1, n_max = 450, tau = 0.1):
    '''
    Base 'analytical' calculation function:
    Underlying function takes single frequency and calculates total temperature contamination (from all n) at that frequency. Steps of this calculation are as follows:
    
    1. Find rest frame of each n
    2. Find redshift where n reaches frequency of interest
    3. Calculates temperature of n at calculated redshift
    4. For all n, if corresponding redshift is not 0 < z < 6, sets temperature to 0.
    5. Adds temperature for all remaining n
    -----------------------------------
    nu [MHz] (float) : Discrete frequency of interest
    -----------------------------------
    returns [K] (float) : Total RRL temperature at frequency, nu
    '''
    #z_space = freqtoz(np.linspace(1420, 200, N))
    n_range = np.arange(n_min, n_max, 1)
    
    #Find rest frame frequency for each n transition
    nu_emitted = rest_freq(n_range)

    #Calculate redshift where each n would be shifted to given nu
    z_contam = (nu_emitted / nu) - 1
    
    #Creates a mask for all redshifts outside of observable band
    z_mask = (z_contam >= 0) & (z_contam <= 6)

    #Calculates temperature for each transition
    temp_line = lt.meanT_shifted(tau, 1, nu_emitted, 20, 1, z_contam)
    
    #Masks out transitions that fall outside of z = (0,6) band
    temp_line = np.where(z_mask, temp_line, 0) #/ z_mask.sum()
    #T_21 = T21(z_contam)
    T_21 = lt.T21(z_contam)
    #print(np.sum(T_21))

    #Combines all n temperatures for given nu
    return(np.sum(temp_line/T_21))

#### Parameters, Testing, and Plotting

In [ ]:
n_min = 30
n_max = 450
HI_band_min = 200
HI_band_max = 1420
N = 10_000

In [ ]:
print('Analytical: Temperature at Center of Frequency Band')
print('avg', aFcn(473, N, n_min, n_max))
#print('min', aFcn(HI_band_max, n_min, n_max))
#print('max', aFcn(HI_band_min, n_min, n_max))

print(T21(2.372563731022383))
print('\n Model: Total Temperature across Frequency Band')
#print(model_plotting(HI_band_min, HI_band_max, N))

In [ ]:
nu_space = ztofreq(np.linspace(0, 6, N))
#nu_space = ztofreq(np.linspace(0, 6, N))
import tauCalc_model as tcm
import lineTemp as lt
import importlib
importlib.reload(tcm)
importlib.reload(lt)

anTemp = np.zeros(N)
anTemp1 = np.zeros(N)
anTemp2 = np.zeros(N)
anTemp3 = np.zeros(N)
anTemp4 = np.zeros(N)
anTemp_carbon = np.zeros(N)
n_space_extended = np.arange(n_min, n_min+571, 1)
tau_new1 = tcm.tau_out(8000, 1e3, 1e5)[1]*1e-6# * np.exp(-tau_C(8000, n_space_extended, 1e5))
tau_new2 = tcm.tau_out(8000, 1e3, 1e6)[1]*1e-6# * np.exp(-tau_C(8000, n_space_extended, 1e6))
tau_new3 = tcm.tau_out(8000, 1e4, 1e7)[1]*1e-6# * np.exp(-tau_C(8000, n_space_extended, 1e7))
tau_new4 = tcm.tau_out(8000, 1e1, 1e4)[1]*1e-6# * np.exp(-tau_C(8000, n_space_extended, 1e7))
#tau_carbon_2 = tcm.tau_out(85, 1e-3, 0.02, i_n = 8)[1]*1e-6
tau_carbon_2 = tcm.tau_out(100, 0.05, 0.02, i_n = 8)[1]*1e-6
tau_carbon = np.concatenate((np.zeros(50), tau_carbon_2))

data_c = np.loadtxt("carbon_tau_combined.txt")
#tau_carbon = data_c[:, 1]*1e-6

for i, nu in enumerate(nu_space):

    anTemp[i] = aFcn2(nu, n_min, n_max, 0.1)
    anTemp1[i] = aFcn2(nu, n_min, n_max, tau_new1[:420])
    anTemp2[i] = aFcn2(nu, n_min, n_max, tau_new2[:420])
    anTemp3[i] = aFcn2(nu, n_min, n_max, tau_new3[:420])
    anTemp4[i] = aFcn2(nu, n_min, n_max, tau_new4[:420])
    '''
    anTemp[i] = lt.aFcn(nu, 0, 6, n_min, n_max, 0.1)
    anTemp1[i] = lt.aFcn(nu, 0, 6, n_min, n_max, tau_new1[:420]) / np.sum(lt.T21((rest_freq(n_space_extended) / nu) - 1))
    anTemp2[i] = lt.aFcn(nu, 0, 6, n_min, n_max, tau_new2[:420]) / np.sum(lt.T21((rest_freq(n_space_extended) / nu) - 1))
    anTemp3[i] = lt.aFcn(nu, 0, 6, n_min, n_max, tau_new3[:420]) / np.sum(lt.T21((rest_freq(n_space_extended) / nu) - 1))
    anTemp4[i] = lt.aFcn(nu, 0, 6, n_min, n_max, tau_new4[:420]) / np.sum(lt.T21((rest_freq(n_space_extended) / nu) - 1))
    '''
    #anTemp_carbon[i] = aFcn_carbon(nu, tau_carbon, 20, 30, 601)
    
    anTemp_carbon[i] = aFcn2(nu, n_min, n_max, tau_carbon[:420])


#### Model and Analytical Plotting Comparison

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots()

secax = ax.secondary_xaxis('top', functions=(ztofreq, freqtoz))
secax.set_xlabel('Frequency [MHz]')
ax.set_xlabel('z [Redshift]')
ax.yaxis.set_ticks_position('both')
#ax.set_ylabel(r'$T_{RRL} ~[\text{K}]$')# ~/~ T_{21cm}$')
ax.set_ylabel(r'$T_{RRL} ~/~ T_{21cm}$')
secax.xaxis.set_tick_params(rotation=70)


secax.set_xticks([220, 300, 400, 600, 800, 1000, 1400])
ax.set_yscale('log')
ax.plot(freqtoz(nu_space), anTemp, label = r'$\tau = 0.1$ Model', color = 'dodgerblue', lw = 3, ls = '--')
ax.plot(freqtoz(nu_space), -anTemp1, label = r"$\tau_{\rm RRL}$ Model", color = "#145da6", alpha = 0.5, lw = 3, ls = '-.')

width = 3

#ax.hlines(model_plotting(200, 300, N, n_min, n_max) , xmin = freqtoz(200), xmax = freqtoz(300), linewidth=2, color='coral', label = 'MWA')
#ax.hlines(model_plotting(500, 1420, N, n_min, n_max) , xmin = freqtoz(500), xmax = freqtoz(1420), linewidth=2, color='pink', label = 'MeerKAT')
ax.hlines(model_plotting(200, 240, N, n_min, n_max) , xmin = freqtoz(200), xmax = freqtoz(240), linewidth=width, color='#AE6F78', label = 'LOFAR')
ax.hlines(model_plotting(200, 1100, N, n_min, n_max) , xmin = freqtoz(200), xmax = freqtoz(1100), linewidth=width, color='#659157', label = 'PUMA')
ax.hlines(model_plotting(300, 1420, N, n_min, n_max) , xmin = freqtoz(300), xmax = freqtoz(1420), linewidth=width, color='#8B1E3F', label = 'CHORD')
ax.hlines(model_plotting(350, 1420, N, n_min, n_max) , xmin = freqtoz(350), xmax = freqtoz(1420), linewidth=width, color='#2B193D', label = 'SKA1-MID')
ax.hlines(model_plotting(400, 800, N, n_min, n_max), xmin = freqtoz(400), xmax = freqtoz(800), linewidth=width, color='#4B8F8C', label = 'CHIME/HIRAX')
ax.hlines(model_plotting(ztofreq(2.5), ztofreq(0), N, n_min, n_max) , xmin = 2.5, xmax = 0, linewidth=width, color='#DA941B', label = 'FAST')
ax.hlines(model_plotting(700, 800, N, n_min, n_max) , xmin = freqtoz(700), xmax = freqtoz(800), linewidth=width, color='red', label = 'TianLai')
ax.hlines(model_plotting(700, 1420, N, n_min, n_max) , xmin = freqtoz(700), xmax = freqtoz(1420), linewidth=width, color='violet', label = 'DSA-2000')
#ax.hlines(model_plotting(200, 250, N, n_min, n_max) , xmin = freqtoz(200), xmax = freqtoz(250), linewidth=2, color='maroon', label = 'HERA')

#ax.hlines(model_plotting(307.5, 345.5, N, n_min, n_max) , xmin = freqtoz(307.5), xmax = freqtoz(345.5), linewidth=2, color='green', label = 'OWFA')



ax.hlines(model_plotting(ztofreq(0.45), ztofreq(0.13), N, n_min, n_max) , xmin = 0.45, xmax = 0.13, linewidth=width, color='#6D3C6C', label = 'BINGO')

plt.ylim(3e-2, 60)
plt.legend(fontsize = 9, loc = 'lower right') #, ncol = 2, bbox_to_anchor=(1, 0.4))
#plt.axvline(2)
#plt.axhline(3)
#plt.savefig('RRL_temp_Surveys_final.png', dpi = 300, bbox_inches='tight')

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots()

secax = ax.secondary_xaxis('top', functions=(ztofreq, freqtoz))
secax.set_xlabel('Frequency [MHz]')
ax.set_xlabel(r'$z_{21}$')
#plt.yticks([1e-3, 1e-2, 1e-1, 1e0, 1e1, 1e2])
ax.yaxis.set_ticks_position('both')
#ax.set_ylabel(r'$T_{RRL} ~[\text{K}]$')# ~/~ T_{21cm}$')
ax.set_ylabel(r'$T_{RRL} ~/~ T_{21cm}$')
secax.xaxis.set_tick_params(rotation=70)


secax.set_xticks([220, 300, 400, 600, 800, 1000, 1400])
ax.set_yscale('log')
ax.plot(freqtoz(nu_space), anTemp,  color = '#837099', lw = 3, ls = '-')#, label = r'$\tau = 0.1$ Model')
ax.plot(freqtoz(nu_space), -anTemp1,  color = "saddlebrown", lw = 3, ls = '-')#label = r"$\tau_{\rm RRL}$ Model",
ax.plot(freqtoz(nu_space), -anTemp2,  color = "#177E89", lw = 3, ls = '-')#label = r"$\tau_{\rm RRL}$ Model",
ax.plot(freqtoz(nu_space), -anTemp3,  color = "goldenrod", lw = 3, ls = '-', zorder = 0)#label = r"$\tau_{\rm RRL}$ Model",
#ax.plot(freqtoz(nu_space), anTemp3, color = "goldenrod", lw = 3, ls = '--', zorder = 2)
ax.plot(freqtoz(nu_space), -anTemp4, color = "crimson", lw = 3, ls = '-', zorder = 0)

ax.plot(freqtoz(nu_space), anTemp_carbon, color = '#FF46E0', lw = 3, ls = '-', zorder = 0)

plt.ylim(1e-6, 100)
plt.xlim(-0.1, 6)



spacing_factor = 1.5  # multiplicative spacing (try 1.15–1.3 to tune)
ymin, ymax = ax.get_ylim()  # get your current y-range (log scale)
base_y = ymin * 1.7         # start a little above the lower limit
width = 3

colors = ["#C89FA6", '#659157', "#B92F58", '#2B193D', "#48C8C2",
          "#FFC531", 'red', 'violet', "#934892", "#ADD803"]
labels = ['LOFAR', 'PUMA', 'CHORD', 'SKA1-MID', 'CHIME/HIRAX',
          'FAST', 'TianLai', 'DSA-2000', 'MeerKAT', 'ASKAP']

xmins = [freqtoz(200), freqtoz(200), freqtoz(300), freqtoz(350), freqtoz(400), 
         2.5, freqtoz(700), freqtoz(700), freqtoz(1000), freqtoz(700)]
xmaxs = [freqtoz(240), freqtoz(1100), freqtoz(1420), freqtoz(1420), freqtoz(800),
          0, freqtoz(800), freqtoz(1420), freqtoz(1420), freqtoz(1420)]

# Define order from bottom (0) to top (len-1)
order = [1, 2, 3, 5, 4, 7, 8, 6, 0, 9]  # example ordering of indices

# Plot evenly spaced horizontal lines in log space using the order
for rank, idx in enumerate(order):
    if idx == 0:
        y = base_y * ((spacing_factor-0.45) ** rank)
    elif idx == 6:
        y = base_y * ((spacing_factor-0.09) ** rank)
    elif idx == 8:
        y = base_y * ((spacing_factor-0.19) ** rank)
    elif idx == 9:
        y = base_y * ((spacing_factor-0.13) ** rank)
    else:
        y = base_y * (spacing_factor ** rank)
    c = colors[idx]
    lbl = labels[idx]
    xmin = xmins[idx]
    xmax = xmaxs[idx]
    ax.hlines(y, xmin=xmin, xmax=xmax, linewidth=width, color=c, label=lbl)
    ax.plot([xmin, xmax], [y]*2, marker='|', color=c, markersize=6, linestyle='None')

import matplotlib.patches as patches

# Example coordinates (replace with your guessed ones)
x0, y0 = -.2, 1.1e-6   # bottom-left corner
width, height = 6.5, 4.5e-5  # rectangle width & height

# Create and add rectangle
rect = patches.Rectangle(
    (x0, y0), width, height,
    linewidth=0, facecolor="#efefff", alpha=0.5, zorder=1  # zorder=0 puts it behind
)
ax.add_patch(rect)
plt.axhline(1, color="#4D4D4D", lw=2, ls='--', label = r'T$_{21cm}$ = T$_{RRL}$')

#leg1 = ax.legend(fontsize = 9, loc = 'center right', ncol = 2, bbox_to_anchor=(1, 0.27), title = '21 cm Surveys', title_fontsize=9)
leg1 = ax.legend(fontsize = 8, loc = 'upper left', bbox_to_anchor=(-0.01, -0.12), ncol = 5, title_fontsize=9, facecolor="#efefff")

from matplotlib.lines import Line2D
model_handles = [
    Line2D([], [], color='#837099', lw=3, ls='-', label=r'PO11'),
    Line2D([], [], color='crimson', lw=3, ls='-', label=r'Model A'),
    Line2D([], [], color='saddlebrown', lw=3, ls='-', label=r'Model B'),
    Line2D([], [], color='#177E89', lw=3, ls='-',  label=r'Model C'),
    Line2D([], [], color='goldenrod', lw=3, ls='-', label=r'Model D'),
    Line2D([], [], color='#FF46E0', lw=3, ls='-', label = r'Model E'),#label='Carbon RRL Model \n n_e = $10^{-1}$, $T_e = 85 K$'),
]

leg2 = ax.legend(handles=model_handles, fontsize=8, bbox_to_anchor=(0.8, 0.9), loc='upper left', title = 'RRL Models', title_fontsize=10, framealpha = 1)
ax.add_artist(leg1)
#ax.axvspan(freqtoz(800), freqtoz(400), color='#48C8C2', alpha=0.3)
'''
ax.axvspan(freqtoz(240), freqtoz(200), color='#AE6F78', alpha=0.3, label='LOFAR')
ax.axvspan(freqtoz(1100), freqtoz(200), color='#659157', alpha=0.3, label='PUMA')
ax.axvspan(freqtoz(1420), freqtoz(300), color='#8B1E3F', alpha=0.3, label='CHORD')
ax.axvspan(freqtoz(1420), freqtoz(350), color='#2B193D', alpha=0.3, label='SKA1-MID')
ax.axvspan(freqtoz(800), freqtoz(400), color='#4B8F8C', alpha=0.3, label='CHIME/HIRAX')
ax.axvspan(0, 2.5, color='#DA941B', alpha=0.3, label='FAST')  # assuming z axis goes left→right
ax.axvspan(freqtoz(800), freqtoz(700), color='red', alpha=0.3, label='TianLai')
ax.axvspan(freqtoz(1420), freqtoz(700), color='violet', alpha=0.3, label='DSA-2000')'''
#plt.savefig('RRL_Final_Code/figures/RRL_temp_Surveys.png', dpi = 300, bbox_inches='tight')
#carbon trace starburst? relation between tauc and tauL


In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots()

secax = ax.secondary_xaxis('top', functions=(ztofreq, freqtoz))
secax.set_xlabel('Frequency [MHz]')
ax.set_xlabel(r'$z_{21}$')
#plt.yticks([1e-3, 1e-2, 1e-1, 1e0, 1e1, 1e2])
ax.yaxis.set_ticks_position('both')
#ax.set_ylabel(r'$T_{RRL} ~[\text{K}]$')# ~/~ T_{21cm}$')
ax.set_ylabel(r'$T_{RRL} ~/~ T_{21cm}$')
secax.xaxis.set_tick_params(rotation=70)


secax.set_xticks([220, 300, 400, 600, 800, 1000, 1400])
ax.set_yscale('log')
ax.plot(freqtoz(nu_space), anTemp,  color = '#837099', lw = 3, ls = '-')#, label = r'$\tau = 0.1$ Model')
ax.plot(freqtoz(nu_space), -anTemp1,  color = "saddlebrown", lw = 3, ls = '-')#label = r"$\tau_{\rm RRL}$ Model",
ax.plot(freqtoz(nu_space), -anTemp2,  color = "#177E89", lw = 3, ls = '-')#label = r"$\tau_{\rm RRL}$ Model",
ax.plot(freqtoz(nu_space), -anTemp3,  color = "goldenrod", lw = 3, ls = '-', zorder = 0)#label = r"$\tau_{\rm RRL}$ Model",
ax.plot(freqtoz(nu_space), anTemp3, color = "goldenrod", lw = 3, ls = '--', zorder = 0)

plt.ylim(1e-6, 100)
plt.xlim(-0.1, 6)



spacing_factor = 1.5  # multiplicative spacing (try 1.15–1.3 to tune)
ymin, ymax = ax.get_ylim()  # get your current y-range (log scale)
base_y = ymin * 1.7         # start a little above the lower limit
width = 3

colors = ['#AE6F78', '#659157', '#8B1E3F', '#2B193D', "red",
          "#FFC531", '#48C8C2', 'violet', '#6D3C6C', "#ADD803"]
labels = ['LOFAR', 'PUMA', 'CHORD', 'SKA1-MID', 'TianLai',
          'MeerKAT', 'CHIME/HIRAX', 'DSA-2000', 'FAST', 'ASKAP']

xmins = [freqtoz(200), freqtoz(200), freqtoz(300), freqtoz(350), freqtoz(700), 
         freqtoz(1000), freqtoz(400), freqtoz(700), 2.5, freqtoz(700)]
xmaxs = [freqtoz(240), freqtoz(1100), freqtoz(1420), freqtoz(1420), freqtoz(800),
          freqtoz(1420), freqtoz(800), freqtoz(1420), 0, freqtoz(1420)]

# Define order from bottom (0) to top (len-1)
order = [1, 2, 3, 5, 4, 
         9, 8, 6, 0, 7]  # example ordering of indices

# Plot evenly spaced horizontal lines in log space using the order
for rank, idx in enumerate(order):
    if idx == 0:
        y = base_y * ((spacing_factor-0.45) ** rank)
    elif idx == 6:
        y = base_y * ((spacing_factor-0.32) ** rank)
    elif idx == 8:
        y = base_y * ((spacing_factor-0.19) ** rank)
    elif idx == 7:
        y = base_y * ((spacing_factor-0.19) ** rank)
    else:
        y = base_y * (spacing_factor ** rank)
    c = colors[idx]
    lbl = labels[idx]
    xmin = xmins[idx]
    xmax = xmaxs[idx]
    ax.hlines(y, xmin=xmin, xmax=xmax, linewidth=width, color=c, label=lbl)
    ax.plot([xmin, xmax], [y]*2, marker='|', color=c, markersize=6, linestyle='None')

import matplotlib.patches as patches

# Example coordinates (replace with your guessed ones)
x0, y0 = -.2, 1.1e-6   # bottom-left corner
width, height = 6.5, 2.5e-5  # rectangle width & height

# Create and add rectangle
rect = patches.Rectangle(
    (x0, y0), width, height,
    linewidth=0, facecolor="#efefff", alpha=1, zorder=1  # zorder=0 puts it behind
)
ax.add_patch(rect)
plt.axhline(1, color="#4D4D4D", lw=2, ls='--')

#leg1 = ax.legend(fontsize = 9, loc = 'center right', ncol = 2, bbox_to_anchor=(1, 0.27), title = '21 cm Surveys', title_fontsize=9)
leg1 = ax.legend(fontsize = 8, loc = 'upper left', bbox_to_anchor=(-0.01, -0.12), ncol = 5, title_fontsize=9, facecolor="#efefff")

from matplotlib.lines import Line2D
model_handles = [
    Line2D([], [], color='#837099', lw=3, ls='-', label=r'$f~\tau_{RRL} = 0.1$ (PO11)'),
    Line2D([], [], color='saddlebrown', lw=3, ls='-', label=r'$n_e = 10^{3},f~\rm EM= 10^{5}$'),
    Line2D([], [], color='#177E89', lw=3, ls='-',  label=r'$n_e = 10^{3},f~\rm EM= 10^{6}$'),
    Line2D([], [], color='goldenrod', lw=3, ls='-', label=r'$n_e = 10^{4},f~\rm EM= 10^{7}$'),
]

leg2 = ax.legend(handles=model_handles, fontsize=9, bbox_to_anchor=(0.64, 0.86), loc='upper left', title=r"$\tau_{\rm RRL}$ Models", title_fontsize=10, framealpha = 1)
ax.add_artist(leg1)
#ax.axvspan(freqtoz(800), freqtoz(400), color='#48C8C2', alpha=0.3)
'''
ax.axvspan(freqtoz(240), freqtoz(200), color='#AE6F78', alpha=0.3, label='LOFAR')
ax.axvspan(freqtoz(1100), freqtoz(200), color='#659157', alpha=0.3, label='PUMA')
ax.axvspan(freqtoz(1420), freqtoz(300), color='#8B1E3F', alpha=0.3, label='CHORD')
ax.axvspan(freqtoz(1420), freqtoz(350), color='#2B193D', alpha=0.3, label='SKA1-MID')
ax.axvspan(freqtoz(800), freqtoz(400), color='#4B8F8C', alpha=0.3, label='CHIME/HIRAX')
ax.axvspan(0, 2.5, color='#DA941B', alpha=0.3, label='FAST')  # assuming z axis goes left→right
ax.axvspan(freqtoz(800), freqtoz(700), color='red', alpha=0.3, label='TianLai')
ax.axvspan(freqtoz(1420), freqtoz(700), color='violet', alpha=0.3, label='DSA-2000')'''
#plt.savefig('RRL_Final_Code/figures/RRL_temp_Surveys.png', dpi = 300, bbox_inches='tight')

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

def assign_lanes(xmins, xmaxs, order=None, pad=0.03):
    """
    Assign each interval to the lowest available horizontal lane.
    Non-overlapping intervals can share a lane.

    pad is a small x-buffer so nearly touching intervals don't visually collide.
    """
    if order is None:
        order = range(len(xmins))

    lanes = []          # stores rightmost x-value currently occupied in each lane
    yvals = [None] * len(xmins)

    for idx in order:
        xlo, xhi = sorted([xmins[idx], xmaxs[idx]])

        placed = False

        for lane_number, lane_xmax in enumerate(lanes):
            if xlo > lane_xmax + pad:
                yvals[idx] = lane_number
                lanes[lane_number] = xhi
                placed = True
                break

        if not placed:
            yvals[idx] = len(lanes)
            lanes.append(xhi)

    return yvals, len(lanes)

fig, (ax, ax_survey) = plt.subplots(
    2, 1,
    sharex=True,
    figsize=(7, 6),
    gridspec_kw={
        "height_ratios": [5, 1.1],
        "hspace": 0.03
    }
)

# -------------------------
# Main RRL plot
# -------------------------
secax = ax.secondary_xaxis('top', functions=(ztofreq, freqtoz))
secax.set_xlabel('Frequency [MHz]')
secax.set_xticks([220, 300, 400, 600, 800, 1000, 1400])
secax.xaxis.set_tick_params(rotation=70)

ax.set_ylabel(r'$\overline{T}_b^{\mathrm{RRL}} ~/~ \overline{T}_b^{21}$')
ax.yaxis.set_ticks_position('both')
ax.set_yscale('log')

ax.plot(freqtoz(nu_space), anTemp,   color='#837099', lw=3, ls='-')
ax.plot(freqtoz(nu_space), -anTemp1, color="saddlebrown", lw=3, ls='-')
ax.plot(freqtoz(nu_space), -anTemp2, color="#177E89", lw=3, ls='-')
ax.plot(freqtoz(nu_space), -anTemp3, color="goldenrod", lw=3, ls='-', zorder=0)
ax.plot(freqtoz(nu_space), -anTemp4, color="crimson", lw=3, ls='-', zorder=0)
ax.plot(freqtoz(nu_space), anTemp_carbon, color='#FF46E0', lw=3, ls='-', zorder=0)

ax.set_ylim(1e-6, 100)
ax.set_xlim(-0.1, 6)

ax.axhline(1, color="#4D4D4D", lw=2, ls='--')

ax.tick_params(labelbottom=False)

# -------------------------
# Bottom survey-coverage plot
# -------------------------
ax_survey.set_facecolor("#efefff")

survey_lw = 2

colors = ["#C89FA6", '#659157', "#B92F58", '#2B193D', "#48C8C2",
          "#FFC531", 'red', 'violet', "#934892", "#ADD803"]

labels = ['LOFAR', 'PUMA', 'CHORD', 'SKA1-MID', 'CHIME/HIRAX',
          'FAST', 'TianLai', 'DSA-2000', 'MeerKAT', 'ASKAP']

xmins = [freqtoz(200), freqtoz(200), freqtoz(300), freqtoz(350), freqtoz(400), 
         2.5, freqtoz(700), freqtoz(700), freqtoz(1000), freqtoz(700)]

xmaxs = [freqtoz(240), freqtoz(1100), freqtoz(1420), freqtoz(1420), freqtoz(800),
         0, freqtoz(800), freqtoz(1420), freqtoz(1420), freqtoz(1420)]

def assign_lanes(xmins, xmaxs, order=None, pad=0.03):
    """
    Assign each interval to the lowest available horizontal lane.
    Non-overlapping intervals can share a lane.
    """
    if order is None:
        order = range(len(xmins))

    lanes = []
    yvals = [None] * len(xmins)

    for idx in order:
        xlo, xhi = sorted([xmins[idx], xmaxs[idx]])

        placed = False

        for lane_number, lane_xmax in enumerate(lanes):
            if xlo > lane_xmax + pad:
                yvals[idx] = lane_number
                lanes[lane_number] = xhi
                placed = True
                break

        if not placed:
            yvals[idx] = len(lanes)
            lanes.append(xhi)

    return yvals, len(lanes)

# This controls which surveys get priority for lower lanes
order = [1, 2, 3, 5, 4, 7, 8, 6, 0, 9]

yvals, nlanes = assign_lanes(xmins, xmaxs, order=order, pad=0.05)

for idx in order:
    y = yvals[idx]

    c = colors[idx]
    lbl = labels[idx]

    xlo, xhi = sorted([xmins[idx], xmaxs[idx]])

    ax_survey.hlines(
        y,
        xmin=xlo,
        xmax=xhi,
        linewidth=survey_lw,
        color=c,
        label=lbl,
        linestyle=(0, (8, 2))
    )

    ax_survey.plot(
        [xlo, xhi], [y, y],
        marker='|',
        color=c,
        markersize=6,
        linestyle='None'
    )




ax_survey.set_ylim(-0.7, nlanes - 0.3)
ax_survey.set_yticks([])
ax_survey.set_xlabel(r'$z_{21}$')

'''
ax_survey.set_ylabel(
    '21 cm\nSurveys',
    rotation=0,
    labelpad=40,
    va='center',
    bbox=dict(facecolor="#efefff", edgecolor="none", alpha=1)
)'''

ax_survey.spines['left'].set_visible(False)
ax_survey.spines['right'].set_visible(False)
ax_survey.spines['top'].set_visible(False)

# -------------------------
# Legends
# -------------------------
'''survey_legend = ax_survey.legend(
    fontsize=8,
    loc='upper center',
    bbox_to_anchor=(0.5, -0.55),
    ncol=5,
    title_fontsize=9,
    facecolor="#efefff"
)
'''
survey_legend = ax_survey.legend(
    fontsize=6,
    ncol=4,
    framealpha=1,
    facecolor="#efefff",
    edgecolor="0.8",
    handlelength=1.3,
    handletextpad=0.4,
    columnspacing=0.7,
    bbox_to_anchor=(0.54, 0.4),
)



model_handles = [
    Line2D([], [], color='#837099', lw=3, ls='-', label=r'PO11 (H)'),
    Line2D([], [], color='crimson', lw=3, ls='-', label=r'Model A (H)'),
    Line2D([], [], color='saddlebrown', lw=3, ls='-', label=r'Model B (H)'),
    Line2D([], [], color='#177E89', lw=3, ls='-', label=r'Model C (H)'),
    Line2D([], [], color='goldenrod', lw=3, ls='-', label=r'Model D (H)'),
    Line2D([], [], color='#FF46E0', lw=3, ls='-', label=r'Model E (C)'),
    Line2D([], [], color="#000000", lw=1.4, ls='--', label=r'$\overline{T}_b^{\mathrm{RRL}} = \overline{T}_b^{21}$'),
]

model_legend = ax.legend(
    handles=model_handles,
    fontsize=8,
    bbox_to_anchor=(0.78, 0.93),
    loc='upper left',
    title='',
    title_fontsize=10,
    framealpha=1
)

#plt.savefig('figures/RRL_temp_Surveys.png', dpi=300, bbox_inches='tight')
plt.show()

## Black Cells

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

def assign_lanes(xmins, xmaxs, order=None, pad=0.03):
    """
    Assign each interval to the lowest available horizontal lane.
    Non-overlapping intervals can share a lane.

    pad is a small x-buffer so nearly touching intervals don't visually collide.
    """
    if order is None:
        order = range(len(xmins))

    lanes = []          # stores rightmost x-value currently occupied in each lane
    yvals = [None] * len(xmins)

    for idx in order:
        xlo, xhi = sorted([xmins[idx], xmaxs[idx]])

        placed = False

        for lane_number, lane_xmax in enumerate(lanes):
            if xlo > lane_xmax + pad:
                yvals[idx] = lane_number
                lanes[lane_number] = xhi
                placed = True
                break

        if not placed:
            yvals[idx] = len(lanes)
            lanes.append(xhi)

    return yvals, len(lanes)

fig, (ax, ax_survey) = plt.subplots(
    2, 1,
    sharex=True,
    figsize=(7, 6),
    gridspec_kw={
        "height_ratios": [5, 1.1],
        "hspace": 0.03
    }
)

# -------------------------
# Main RRL plot
# -------------------------
secax = ax.secondary_xaxis('top', functions=(ztofreq, freqtoz))
secax.set_xlabel('Frequency [MHz]')
secax.set_xticks([220, 300, 400, 600, 800, 1000, 1400])
secax.xaxis.set_tick_params(rotation=70)

ax.set_ylabel(r'$\overline{T}_b^{\mathrm{RRL}} ~/~ \overline{T}_b^{21}$')
ax.yaxis.set_ticks_position('both')
ax.set_yscale('log')

ax.plot(freqtoz(nu_space), anTemp,   color='#837099', lw=3, ls='-')
ax.plot(freqtoz(nu_space), -anTemp1, color="saddlebrown", lw=3, ls='-')
ax.plot(freqtoz(nu_space), -anTemp2, color="#177E89", lw=3, ls='-')
ax.plot(freqtoz(nu_space), -anTemp3, color="goldenrod", lw=3, ls='-', zorder=0)
ax.plot(freqtoz(nu_space), -anTemp4, color="crimson", lw=3, ls='-', zorder=0)
ax.plot(freqtoz(nu_space), anTemp_carbon, color='#FF46E0', lw=3, ls='-', zorder=0)

ax.set_ylim(1e-6, 100)
ax.set_xlim(-0.1, 6)

ax.axhline(1, color="#4D4D4D", lw=2, ls='--')

ax.tick_params(labelbottom=False)

# -------------------------
# Bottom survey-coverage plot
# -------------------------
ax_survey.set_facecolor("#efefff")

survey_lw = 1

colors = ["#C89FA6", '#659157', "#B92F58", '#2B193D', "#48C8C2",
          "#FFC531", 'red', 'violet', "#934892", "#ADD803"]

labels = ['LOFAR', 'PUMA', 'CHORD', 'SKA1-MID', 'CHIME/HIRAX',
          'FAST', 'TianLai', 'DSA-2000', 'MeerKAT', 'ASKAP']

xmins = [freqtoz(200), freqtoz(200), freqtoz(300), freqtoz(350), freqtoz(400), 
         2.5, freqtoz(700), freqtoz(700), freqtoz(1000), freqtoz(700)]

xmaxs = [freqtoz(240), freqtoz(1100), freqtoz(1420), freqtoz(1420), freqtoz(800),
         0, freqtoz(800), freqtoz(1420), freqtoz(1420), freqtoz(1420)]

def assign_lanes(xmins, xmaxs, order=None, pad=0.03):
    """
    Assign each interval to the lowest available horizontal lane.
    Non-overlapping intervals can share a lane.
    """
    if order is None:
        order = range(len(xmins))

    lanes = []
    yvals = [None] * len(xmins)

    for idx in order:
        xlo, xhi = sorted([xmins[idx], xmaxs[idx]])

        placed = False

        for lane_number, lane_xmax in enumerate(lanes):
            if xlo > lane_xmax + pad:
                yvals[idx] = lane_number
                lanes[lane_number] = xhi
                placed = True
                break

        if not placed:
            yvals[idx] = len(lanes)
            lanes.append(xhi)

    return yvals, len(lanes)

# This controls which surveys get priority for lower lanes
order = [1, 2, 3, 5, 4, 7, 8, 6, 0, 9]

yvals, nlanes = assign_lanes(xmins, xmaxs, order=order, pad=0.05)

for idx in order:
    y = yvals[idx]

    c = colors[idx]
    lbl = labels[idx]

    xlo, xhi = sorted([xmins[idx], xmaxs[idx]])

    ax_survey.hlines(
        y,
        xmin=xlo,
        xmax=xhi,
        linewidth=survey_lw,
        color=c,
        label=lbl
    )

    ax_survey.plot(
        [xlo, xhi], [y, y],
        marker='|',
        color=c,
        markersize=6,
        linestyle='None'
    )

ax_survey.set_ylim(-0.7, nlanes - 0.3)
ax_survey.set_yticks([])
ax_survey.set_xlabel(r'$z_{21}$')

ax_survey.spines['left'].set_visible(False)
ax_survey.spines['right'].set_visible(False)
ax_survey.spines['top'].set_visible(False)

# -------------------------
# Legends
# -------------------------
survey_legend = ax_survey.legend(
    fontsize=8,
    loc='upper center',
    bbox_to_anchor=(0.5, -0.55),
    ncol=5,
    title_fontsize=9,
    facecolor="#efefff"
)

model_handles = [
    Line2D([], [], color='#837099', lw=3, ls='-', label=r'PO11'),
    Line2D([], [], color='crimson', lw=3, ls='-', label=r'Model A'),
    Line2D([], [], color='saddlebrown', lw=3, ls='-', label=r'Model B'),
    Line2D([], [], color='#177E89', lw=3, ls='-', label=r'Model C'),
    Line2D([], [], color='goldenrod', lw=3, ls='-', label=r'Model D'),
    Line2D([], [], color='#FF46E0', lw=3, ls='-', label=r'Model E'),
]

model_legend = ax.legend(
    handles=model_handles,
    fontsize=8,
    bbox_to_anchor=(0.8, 0.9),
    loc='upper left',
    title='RRL Models',
    title_fontsize=10,
    framealpha=1
)

# Make everything white for black background
ax.tick_params(axis='both', colors='white')
secax.tick_params(axis='both', colors='white')
ax_survey.tick_params(axis='both', colors='white')

ax.xaxis.label.set_color('white')
ax.yaxis.label.set_color('white')
ax_survey.xaxis.label.set_color('white')
secax.xaxis.label.set_color('white')

for spine in ax.spines.values():
    spine.set_color('white')
for spine in secax.spines.values():
    spine.set_color('white')

# Make figure + axes transparent
fig.patch.set_alpha(0)
ax.set_facecolor('none')
secax.set_facecolor('none')

#fig.patch.set_alpha(1)
#fig.patch.set_facecolor("black")

#ax.set_facecolor("black")
#ax_survey.set_facecolor("black")

#plt.savefig('figures/RRL_temp_Surveys.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots()

secax = ax.secondary_xaxis('top', functions=(ztofreq, freqtoz))
secax.set_xlabel('Frequency [MHz]')
ax.set_xlabel(r'$z_{21}$')
#plt.yticks([1e-3, 1e-2, 1e-1, 1e0, 1e1, 1e2])
ax.yaxis.set_ticks_position('both')
#ax.set_ylabel(r'$T_{RRL} ~[\text{K}]$')# ~/~ T_{21cm}$')
ax.set_ylabel(r'$T_{RRL} ~/~ T_{21cm}$')
secax.xaxis.set_tick_params(rotation=70)


secax.set_xticks([220, 300, 400, 600, 800, 1000, 1400])
ax.set_yscale('log')
ax.plot(freqtoz(nu_space), anTemp,  color = '#837099', lw = 3, ls = '-')#, label = r'$\tau = 0.1$ Model')
ax.plot(freqtoz(nu_space), -anTemp1,  color = "saddlebrown", lw = 3, ls = '-')#label = r"$\tau_{\rm RRL}$ Model",
ax.plot(freqtoz(nu_space), -anTemp2,  color = "#177E89", lw = 3, ls = '-')#label = r"$\tau_{\rm RRL}$ Model",
ax.plot(freqtoz(nu_space), -anTemp3,  color = "goldenrod", lw = 3, ls = '-', zorder = 0)#label = r"$\tau_{\rm RRL}$ Model",
ax.plot(freqtoz(nu_space), anTemp3, color = "goldenrod", lw = 3, ls = '--', zorder = 0)

plt.ylim(1e-6, 100)
plt.xlim(-0.1, 6)



spacing_factor = 1.5  # multiplicative spacing (try 1.15–1.3 to tune)
ymin, ymax = ax.get_ylim()  # get your current y-range (log scale)
base_y = ymin * 1.7         # start a little above the lower limit
width = 3

colors = ['#AE6F78', '#659157', '#8B1E3F', '#2B193D', "#48C8C2",
          "#FFC531", 'red', 'violet', '#6D3C6C']
labels = ['LOFAR', 'PUMA', 'CHORD', 'SKA1-MID', 'CHIME/HIRAX',
          'FAST', 'TianLai', 'DSA-2000', 'BINGO']

xmins = [freqtoz(200), freqtoz(200), freqtoz(300), freqtoz(350),
         freqtoz(400), 2.5, freqtoz(700), freqtoz(700), 0.45]
xmaxs = [freqtoz(240), freqtoz(1100), freqtoz(1420), freqtoz(1420),
         freqtoz(800), 0, freqtoz(800), freqtoz(1420), 0.13]

# Define order from bottom (0) to top (len-1)
order = [1, 2, 3, 5, 4, 7, 8, 6, 0]  # example ordering of indices

# Plot evenly spaced horizontal lines in log space using the order
for rank, idx in enumerate(order):
    if idx == 0:
        y = base_y * ((spacing_factor-0.45) ** rank)
    elif idx == 6:
        y = base_y * ((spacing_factor-0.09) ** rank)
    elif idx == 8:
        y = base_y * ((spacing_factor-0.19) ** rank)
    else:
        y = base_y * (spacing_factor ** rank)
    c = colors[idx]
    lbl = labels[idx]
    xmin = xmins[idx]
    xmax = xmaxs[idx]
    ax.hlines(y, xmin=xmin, xmax=xmax, linewidth=width, color=c, label=lbl)
    ax.plot([xmin, xmax], [y]*2, marker='|', color=c, markersize=6, linestyle='None')

import matplotlib.patches as patches

# Example coordinates (replace with your guessed ones)
x0, y0 = -.2, 1.1e-6   # bottom-left corner
width, height = 6.5, 2.5e-5  # rectangle width & height

# Create and add rectangle
rect = patches.Rectangle(
    (x0, y0), width, height,
    linewidth=0, facecolor="#efefff", alpha=1, zorder=1  # zorder=0 puts it behind
)
ax.add_patch(rect)

#leg1 = ax.legend(fontsize = 9, loc = 'center right', ncol = 2, bbox_to_anchor=(1, 0.27), title = '21 cm Surveys', title_fontsize=9)
leg1 = ax.legend(fontsize = 9, loc = 'center right', ncol = 1, bbox_to_anchor=(1, 0.6), title = '21 cm Surveys', title_fontsize=9)

from matplotlib.lines import Line2D
model_handles = [
    Line2D([], [], color='#837099', lw=3, ls='-', label=r'$f~\tau_{RRL} = 0.1$'),
    Line2D([], [], color='saddlebrown', lw=3, ls='-', label=r'$n_e = 10^{3},f~\rm EM= 10^{5}$'),
    Line2D([], [], color='#177E89', lw=3, ls='-',  label=r'$n_e = 10^{3},f~\rm EM= 10^{6}$'),
    Line2D([], [], color='goldenrod', lw=3, ls='-', label=r'$n_e = 10^{4},f~\rm EM= 10^{7}$'),
]

leg2 = ax.legend(handles=model_handles, fontsize=9, loc='upper left', title=r"$\tau_{\rm RRL}$ Models", title_fontsize=10)
ax.add_artist(leg1)
#ax.axvspan(freqtoz(800), freqtoz(400), color='#48C8C2', alpha=0.3)
'''
ax.axvspan(freqtoz(240), freqtoz(200), color='#AE6F78', alpha=0.3, label='LOFAR')
ax.axvspan(freqtoz(1100), freqtoz(200), color='#659157', alpha=0.3, label='PUMA')
ax.axvspan(freqtoz(1420), freqtoz(300), color='#8B1E3F', alpha=0.3, label='CHORD')
ax.axvspan(freqtoz(1420), freqtoz(350), color='#2B193D', alpha=0.3, label='SKA1-MID')
ax.axvspan(freqtoz(800), freqtoz(400), color='#4B8F8C', alpha=0.3, label='CHIME/HIRAX')
ax.axvspan(0, 2.5, color='#DA941B', alpha=0.3, label='FAST')  # assuming z axis goes left→right
ax.axvspan(freqtoz(800), freqtoz(700), color='red', alpha=0.3, label='TianLai')
ax.axvspan(freqtoz(1420), freqtoz(700), color='violet', alpha=0.3, label='DSA-2000')'''
#plt.savefig('RRL_temp_Surveys_final.png', dpi = 300, bbox_inches='tight')

# Make everything white for black background
ax.tick_params(axis='both', colors='white')
secax.tick_params(axis='both', colors='white')

ax.xaxis.label.set_color('white')
ax.yaxis.label.set_color('white')
secax.xaxis.label.set_color('white')

for spine in ax.spines.values():
    spine.set_color('white')
for spine in secax.spines.values():
    spine.set_color('white')

# Make figure + axes transparent
fig.patch.set_alpha(0)
ax.set_facecolor('none')
secax.set_facecolor('none')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# ---------- user: keep your ztofreq / freqtoz definitions ----------
# def ztofreq(z): ...
# def freqtoz(freq): ...
# ------------------------------------------------------------------

fig, ax = plt.subplots(figsize=(12, 4))

# bottom axis label
ax.set_xlabel(r'$z_{21}$', fontsize = 16)
ax.set_yticks([])              # remove y tick marks / numbers
ax.spines['left'].set_visible(False)
ax.spines['right'].set_visible(False)

# secondary top axis in Frequency [MHz]
secax = ax.secondary_xaxis('top', functions=(ztofreq, freqtoz))
secax.set_xlabel('Frequency [MHz]', fontsize = 16)
secax.xaxis.set_tick_params(rotation=70)

# set ticks you showed earlier
secax.set_xticks([220, 300, 400, 600, 800, 1000, 1400])
ax.set_xticks([0, 1, 2, 3, 4, 5, 6])

# bottom xticks
ax.tick_params(axis='x', labelsize=14)

# top xticks (secondary axis)
secax.tick_params(axis='x', labelsize=14)


'''

spacing_factor = 1.5  # multiplicative spacing (try 1.15–1.3 to tune)
ymin, ymax = ax.get_ylim()  # get your current y-range (log scale)
base_y = ymin * 1.7         # start a little above the lower limit
width = 3

colors = ["#C89FA6", '#659157', "#B92F58", '#2B193D', "#48C8C2",
          "#FFC531", 'red', 'violet', "#934892", "#ADD803"]
labels = ['LOFAR', 'PUMA', 'CHORD', 'SKA1-MID', 'CHIME/HIRAX',
          'FAST', 'TianLai', 'DSA-2000', 'MeerKAT', 'ASKAP']

xmins = [freqtoz(200), freqtoz(200), freqtoz(300), freqtoz(350), freqtoz(400), 
         2.5, freqtoz(700), freqtoz(700), freqtoz(1000), freqtoz(700)]
xmaxs = [freqtoz(240), freqtoz(1100), freqtoz(1420), freqtoz(1420), freqtoz(800),
          0, freqtoz(800), freqtoz(1420), freqtoz(1420), freqtoz(1420)]

# Define order from bottom (0) to top (len-1)
order = [1, 2, 3, 5, 4, 7, 8, 6, 0, 9]  # example ordering of indices

# Plot evenly spaced horizontal lines in log space using the order
for rank, idx in enumerate(order):
    if idx == 0:
        y = base_y * ((spacing_factor-0.45) ** rank)
    elif idx == 6:
        y = base_y * ((spacing_factor-0.09) ** rank)
    elif idx == 8:
        y = base_y * ((spacing_factor-0.19) ** rank)
    elif idx == 9:
        y = base_y * ((spacing_factor-0.13) ** rank)
    else:
        y = base_y * (spacing_factor ** rank)
    c = colors[idx]
    lbl = labels[idx]
    xmin = xmins[idx]
    xmax = xmaxs[idx]
    ax.hlines(y, xmin=xmin, xmax=xmax, linewidth=width, color=c, label=lbl)
    ax.plot([xmin, xmax], [y]*2, marker='|', color=c, markersize=6, linestyle='None')
'''







# --- experiment data (use your given lists) ---
colors = ["#FE5874", '#659157', "#B92FB4", "#FFFFFF", "#48C8C2",
          "#FFC531", "#FF002B", "#2C76F5", "#E24FE0", "#ADD803"]
labels = ['LOFAR', 'PUMA', 'CHORD', 'SKA1-MID', 'CHIME/HIRAX',
          'FAST', 'TianLai', 'DSA-2000', 'MeerKAT', 'ASKAP']

xmins = [freqtoz(200), freqtoz(200), freqtoz(300), freqtoz(350), freqtoz(400), 
         2.5, freqtoz(700), freqtoz(700), freqtoz(1000), freqtoz(700)]
xmaxs = [freqtoz(240), freqtoz(1100), freqtoz(1420), freqtoz(1420), freqtoz(800),
          0, freqtoz(800), freqtoz(1420), freqtoz(1420), freqtoz(1420)]

# choose ordering for vertical stacking (bottom -> top)
order = [1, 2, 3, 5, 4, 7, 8, 6, 0, 9]

# compute nice, evenly spaced y positions in the visible band (0.1..0.9)
n = len(labels)
y_positions = np.linspace(0.15, 0.85, n)

# Plot each experiment as a horizontal bar at its assigned y
bar_height = 0.02  # visual thickness of the band
# --- experiment bands plotted as before ---
xlim = ax.get_xlim()

# Create a right-hand margin for labels
label_x = xlim[1] + 0.01 * (xlim[1] - xlim[0])  # adjust if needed

for rank, idx in enumerate(order):
    y = y_positions[rank]
    xmin = xmins[idx]
    xmax = xmaxs[idx]
    c = colors[idx]
    lbl = labels[idx]

    # draw the line
    ax.hlines(y, xmin=xmin, xmax=xmax, linewidth=6, color=c, zorder=3)
    ax.plot([xmin, xmax], [y, y], marker='|', color=c, markersize=8,
            linestyle='None', zorder=4)

    # ---- NEW: large, non-overlapping labels placed to the right ----
    ax.text(label_x, y, lbl,
            va='center', ha='left',
            fontsize=14, weight='regular',  # bigger font
            color='white',               # readable always
            zorder=10)


# optional: a faint background rectangle (like you had) to highlight the band region
# coords chosen to sit behind the bands — adjust if necessary
x0 = -0.5
rect_width = 7.0
rect_y0 = 0.1
rect_height = 0.8
rect = patches.Rectangle((x0, rect_y0), rect_width, rect_height,
                         linewidth=0, facecolor="#efefff", alpha=0.25, zorder=0)
ax.add_patch(rect)

# tighten display and make it look clean
ax.set_ylim(0, 1)
ax.set_xlim(0, 6)
ax.set_frame_on(False)

plt.tight_layout()
# Make everything white for black background
ax.tick_params(axis='x', colors='white')
secax.tick_params(axis='x', colors='white')

ax.xaxis.label.set_color('white')
secax.xaxis.label.set_color('white')

for spine in ax.spines.values():
    spine.set_color('white')
for spine in secax.spines.values():
    spine.set_color('white')

# Make figure + axes transparent
fig.patch.set_alpha(0)
ax.set_facecolor('none')
secax.set_facecolor('none')

plt.show()


# Rough Carbon Estimate

In [ ]:
def rest_freq_carbon(n):
    '''
    Calculates H transition frequency in MHz
    ------------------
    n            (int)   : Transition level
    ------------------
    return [MHz] (float) : Frequency of n transition
    '''
    return(((1 / (n)**2) - (1 / (n+1)**2)) * 1.1e5 * 3e10 * 1e-6)

def madauFit(z):
    '''
    Calculates the fit for SFR from redshift (Madau & Dickinson 2014)
    -----------------------------------
    z                                 (array) : Redshift 
    -----------------------------------
    returns [Solar Mass / yr / Mpc^3] (array) : Star formation rate Density
    '''
    return (0.015 * (1 + z)**2.7 / (1 + ((1 + z)/2.9)**5.6))

def freq_z(nu_emit, z):
    '''
    Calculates cosmological redshift of ground state H frequency
    ------------------
    nu_emit (float) : Ground state frequency of transition
    z       (array) : Redshift space of 21cm emission
    ------------------
    return  (array) : Redshifted frequency of H transition
    '''
    return(nu_emit/(1+z))

def meanT_shifted_carbon(tau_L, N_HII, nu_emit, linewidth, z):
    '''
    Calculates the Line temperature of each RRL at the associated redshift. This function is only passed those RRLs (and associated redshifts) that are observable by 
    the 21cm observing experiment.
    ---------------------------------------
    E_SFR     [Solar Mass / yr / Mpc^3] (float) : Comoving Star Formation Rate
    tau_L                               (float) : Optical Depth of RRL in HII region
    N_HII                               (float) : Number of HII obscuring regions w/ optical depth, tau_L
    nu_emit   [MHz]                     (float) : Rest frame frequency of RRL
    linewidth [km / s]                  (float) : Linewidth of RRL, produced by HII region
    bandwidth [MHz]                     (float) : Bandwidth of 21cm observation
    z                                   (array) : Redshift where RRL is observable
    --------------------------------------
    return    [K]                       (array) : Line temperature of particular RRL
    '''
    E_SFR = madauFit(z)
    nu_shift = freq_z(nu_emit, z)
    
    T_L = 1.8e-6 * (E_SFR / 0.1) * ((tau_L * N_HII) / 0.1) * (nu_shift / 500)**(-2.8) * (linewidth / 20) * ((1+z)/3)**(-2.3)
    return(T_L)


def aFcn_carbon(nu, n_min = 1, n_max = 450, tau_new=0.1):
    '''
    Base 'analytical' calculation function:
    Underlying function takes single frequency and calculates total temperature contamination (from all n) at that frequency. Steps of this calculation are as follows:
    
    1. Find rest frame of each n
    2. Find redshift where n reaches frequency of interest
    3. Calculates temperature of n at calculated redshift
    4. For all n, if corresponding redshift is not 0 < z < 6, sets temperature to 0.
    5. Adds temperature for all remaining n
    -----------------------------------
    nu [MHz] (float) : Discrete frequency of interest
    -----------------------------------
    returns [K] (float) : Total RRL temperature at frequency, nu
    '''
    
    n_range = np.arange(n_min, n_max, 1)
    
    #Find rest frame frequency for each n transition
    nu_emitted = rest_freq_carbon(n_range)

    #Calculate redshift where each n would be shifted to given nu
    z_contam = (nu_emitted / nu) - 1
    
    #Creates a mask for all redshifts outside of observable band
    z_mask = (z_contam >= 0) & (z_contam <= 6)

    #Calculates temperature for each transition
    temp_line = meanT_shifted_carbon(tau_new, 1, nu_emitted, 20, z_contam)
    
    #Masks out transitions that fall outside of z = (0,6) band
    temp_line = np.where(z_mask, temp_line, 0)

    T_21 = lt.T21(z_contam)

    #Combines all n temperatures for given nu
    return(np.sum(temp_line/T_21))

def freqtoz(x):
    return((1420 / x) - 1)
def ztofreq(x):
    return (1420 / (1+x))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import lineTemp as lt
import tauCalc_model as tcm
import importlib
importlib.reload(lt)
importlib.reload(tcm)
T = 110
ne = 0.06
linewidth = 10
n = 500
tau_peak = 2.2e-4
N = 10000
#intTau = np.sqrt(np.pi)/(2*np.sqrt(np.log(2))) * tau_peak * linewidth

#intTau_carbon = np.absolute(tcm.tau_out(100, 1e-2, 0.02)[1]*1e-6)


nu_space = ztofreq(np.linspace(0, 6, N))
#test = aFcn_carbon(1420, intTau, linewidth)
#plt.plot(intTau_carbon, label = 'tau_c')

anTemp_carbon = np.zeros(N)

data_c = np.loadtxt("carbon_tau_combined.txt")
n_carbon = data_c[:-1, 0]
tau_carbon = data_c[:-1, 1]*1e-6
plt.plot(n_carbon, tau_carbon, label = 'tau_c')


for i, nu in enumerate(nu_space):
    anTemp_carbon[i] = aFcn_carbon(nu, 30, 600, tau_carbon)


#Discovery of carbon radio recombination lines in absorption towards Cygnus A

fig, ax = plt.subplots()

secax = ax.secondary_xaxis('top', functions=(ztofreq, freqtoz))
secax.set_xlabel('Frequency [MHz]')
ax.set_xlabel(r'$z_{21}$')
#plt.yticks([1e-3, 1e-2, 1e-1, 1e0, 1e1, 1e2])
ax.yaxis.set_ticks_position('both')
#ax.set_ylabel(r'$T_{RRL} ~[\text{K}]$')# ~/~ T_{21cm}$')
ax.set_ylabel(r'$T_{RRL} ~/~ T_{21cm}$')
secax.xaxis.set_tick_params(rotation=70)
ax.plot(freqtoz(nu_space), -anTemp_carbon)

secax.set_xticks([220, 300, 400, 600, 800, 1000, 1400])
ax.set_yscale('log')
#plt.ylim(1e-6, 100)
plt.xlim(-0.1, 6)
print(anTemp_carbon)
